# `paper_tmlr_1` SPLM em_ln γ-sweep diagnostic — Colab harness (H100)

Sweeps the SPLM em_ln **fixed damping coefficient γ** over five values at the E9 scale-up configuration to test the hypothesis that the small-scale γ⋆ ≈ 0.166 (E5 winner @ D=128, L=4) inflates by ~1.8× at scaleup to ≈ 0.30 (the colab_pilot Arm 2 winner).

## γ values

| γ | rationale |
|---:|---|
| 0.166 | small-scale γ⋆ (E5 winner) — does it transfer? |
| 0.20  | small + ε |
| 0.25  | midpoint |
| 0.30  | colab_pilot Arm 2 winner — argmin val PPL? |
| 0.35  | over-damped — should under-perform if 0.30 is the optimum |

Single seed (seed=0) per γ. The pilot already showed γ-sensitivity is small at scaleup, so n=1 is sufficient for an argmin diagnostic; if the answer is ambiguous (top two PPLs within 0.20 of each other), follow up with a second seed on those γ values.

## Wall-clock estimates (H100 80 GB)

| γ | per-cell |
|---:|---:|
| 0.166 | ~12 min |
| 0.20  | ~12 min |
| 0.25  | ~12 min |
| 0.30  | ~12 min |
| 0.35  | ~12 min |
| **Total** | **~60 min** |

Resume-friendly: each γ-cell skips if its `*_summary.md` already exists in Drive (tag pattern: `splm_em_ln_scaleup_scaleup_g{int(γ*1000):03d}_seed0`, e.g. `g166_seed0`, `g300_seed0`).

## Decision rule

- argmin(γ → val PPL) ⇒ scaleup γ⋆.
- scaleup multiplier = γ⋆_scaleup / 0.166.  Predicted ≈ 1.8×; confirmed if multiplier ∈ [1.55, 2.05].
- if PPL range across the sweep < 0.20 PPL: γ is **insensitive at scaleup** (the choice within 0.166–0.35 does not change the architectural conclusion).


## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time
from pathlib import Path

# Set the CUDA allocator config BEFORE any torch import or CUDA context
# creation.  expandable_segments=True helps the caching allocator grow
# and shrink segments to reduce fragmentation.  This is propagated to
# every trainer subprocess we launch via subprocess.Popen, which
# inherits the parent process's environment.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
print('PYTORCH_ALLOC_CONF =', os.environ['PYTORCH_ALLOC_CONF'])

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/semsimula_pilot')
    REPO_PARENT  = Path('/content')
else:
    DRIVE_ROOT   = Path.home() / 'semsimula_pilot'
    REPO_PARENT  = Path.cwd().parent.parent.parent.parent  # local fallback

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_LOGS    = DRIVE_ROOT / 'logs'
DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)
print('Logs dir     :', DRIVE_LOGS)


In [ ]:
# Clone (or pull) the semsimula repo into Colab's ephemeral disk.
# Replace REPO_URL with your fork or branch as needed.
REPO_URL  = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_NAME = 'semsimula'
REPO_DIR  = REPO_PARENT / REPO_NAME

if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL} -> {REPO_DIR} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR} already exists; pulling latest ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

SCALEUP_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
assert SCALEUP_DIR.exists(), f'scaleup dir not found at {SCALEUP_DIR}'
print('scaleup dir  :', SCALEUP_DIR)
print('contents     :', sorted(p.name for p in SCALEUP_DIR.iterdir())[:25])


In [ ]:
# Install Python dependencies. Colab already ships with a recent torch+CUDA;
# we only need to top up the smaller helpers.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'pyarrow'], check=True)
print('Dependencies OK')


In [ ]:
# Verify GPU + report device specs.  H100 (80 GB) recommended for the
# tightest wall-clock; A100 (40 GB) and L4 (24 GB) also work but slower.
import torch
print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'GPU         : {dev.name}')
    print(f'Total VRAM  : {dev.total_memory / 1e9:.1f} GB')
    print(f'CUDA cap    : sm_{dev.major}{dev.minor}')
    print(f'CUDA driver : {torch.version.cuda}')
    if 'H100' not in dev.name:
        print('NOTE: wall-clock estimates assume H100; expect ~3x slower on A100, ~5x slower on L4.')
else:
    print('NO CUDA GPU - switch to a GPU runtime: Runtime > Change runtime type > GPU (H100 recommended)')
    raise SystemExit(1)


In [ ]:
# Verify data + logfreq surprisal files are present (tracked in repo).
DATA_DIR     = REPO_DIR / 'notebooks' / 'conservative_arch' / 'data'
LOGFREQ_PATH = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
NPZ_PATH     = DATA_DIR / 'tinystories_gpt2_1files_5000000toks.npz'

assert NPZ_PATH.exists(),    f'TinyStories .npz missing at {NPZ_PATH}'
assert LOGFREQ_PATH.exists(), f'logfreq .npy missing at {LOGFREQ_PATH}'
print('TinyStories .npz   :', NPZ_PATH, f'({NPZ_PATH.stat().st_size/1e6:.1f} MB)')
print('logfreq surprisal  :', LOGFREQ_PATH, f'({LOGFREQ_PATH.stat().st_size/1e3:.1f} KB)')


## 2. Run-arm helper (with `tag_suffix_override` for γ-sweep)

In [ ]:
import datetime as _dt

def _summary_exists_for(prefix: str, suffix: str = '') -> Path | None:
    """Look for any *_summary.md whose name starts with `prefix` and
    contains `suffix` as a substring. Returns the matching path or None.
    Uses a single-asterisk glob (Python 3.12 rejects adjacent '**' unless
    it is an entire path component) and filters the suffix in Python."""
    for p in sorted(DRIVE_RESULTS.glob(f'{prefix}*_summary.md')):
        if suffix and suffix not in p.name:
            continue
        return p
    return None


def run_arm(name: str, script: str, args: list[str], skip_prefix: str,
            seed: int = 0, log_label: str | None = None,
            extra_skip_suffix: str = '',
            tag_suffix_override: str | None = None) -> int:
    """Launch a single training arm via subprocess.

    Args:
      name              : human-readable name for logging
      script            : trainer script filename inside SCALEUP_DIR
      args              : list of extra CLI args to pass
      skip_prefix       : prefix used to detect a completed run; if any
                          file in DRIVE_RESULTS matches
                          `{skip_prefix}*_summary.md` we skip.
      seed              : seed value (added to args automatically)
      log_label         : optional label for the Drive log file; defaults
                          to `name` lowercased.
      extra_skip_suffix : additional substring required in the matched
                          summary filename (for arms that share a prefix).
      tag_suffix_override : if not None, replaces the default
                          `seed{seed}` tag-suffix string.  Used by the
                          γ-sweep dispatcher to encode γ in the artifact
                          filename.
    """
    label = log_label or name.lower().replace(' ', '_').replace('/', '_')
    existing = _summary_exists_for(skip_prefix, extra_skip_suffix)
    if existing is not None:
        print(f'[run] {name}: SKIP (found {existing.name})')
        return 0
    tag_suffix = tag_suffix_override if tag_suffix_override is not None else f'seed{seed}'
    cmd = [sys.executable, '-u', str(SCALEUP_DIR / script),
           '--mode', 'scaleup',
           '--seed', str(seed),
           '--results-dir', str(DRIVE_RESULTS),
           '--tag-suffix', tag_suffix,
           ] + list(args)
    ts = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path = DRIVE_LOGS / f'{label}_{tag_suffix}_{ts}.log'
    print(f'[run] {name}')
    print(f'      cmd: {" ".join(cmd)}')
    print(f'      log: {log_path}')
    t0 = time.time()
    with log_path.open('w') as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True,
                                bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            logf.write(line)
            logf.flush()
        rc = proc.wait()
    dt = (time.time() - t0) / 3600.0
    print(f'[run] {name}: exit={rc}  elapsed={dt:.2f} h')
    return rc


def run_arm_seed(name: str, script: str, args: list[str],
                 skip_prefix_base: str, seed: int, **kwargs) -> int:
    """Like `run_arm` but `skip_prefix_base` is auto-extended with
    `_seed{seed}` so the skip-if-done check is per-seed (otherwise
    seed-1 would be skipped just because seed-0 already wrote a
    summary file with the same prefix)."""
    return run_arm(
        name=f'{name} seed={seed}',
        script=script,
        args=args,
        skip_prefix=f'{skip_prefix_base}_seed{seed}',
        seed=seed,
        **kwargs,
    )


## 3. γ-sweep dispatch

Each cell launches one γ value with `--fixed-gamma {γ}` and a tag suffix `g{int(γ*1000):03d}_seed{seed}` so the per-γ artifacts coexist in Drive.  Run them in any order; the aggregator picks up all five by globbing for the tag pattern.


In [ ]:
SEED   = 0
GAMMAS = [0.166, 0.20, 0.25, 0.30, 0.35]
print('seed   :', SEED)
print('gammas :', GAMMAS)


def gamma_tag(gamma: float) -> str:
    """Encode γ as the suffix used by the trainer's tag.
    e.g. 0.30 -> 'g300', 0.166 -> 'g166'."""
    return f'g{int(round(gamma * 1000)):03d}'


def run_gamma(gamma: float, seed: int = SEED) -> int:
    gtag = gamma_tag(gamma)
    return run_arm(
        name=f'SPLM em_ln γ={gamma:.3f}',
        script='train_splm_em_ln_scaleup.py',
        args=['--fixed-gamma', f'{gamma:.4f}'],
        skip_prefix=f'splm_em_ln_scaleup_scaleup_{gtag}_seed{seed}',
        seed=seed,
        log_label=f'splm_em_ln_{gtag}',
        tag_suffix_override=f'{gtag}_seed{seed}',
    )


In [ ]:
# γ-sweep dispatch (~60 min total on H100)
for gamma in GAMMAS:
    rc = run_gamma(gamma)
    assert rc == 0, f'SPLM em_ln gamma={gamma:.3f} returned exit {rc}'


## 4. Aggregate γ-sweep results

Reads the five per-γ checkpoints, identifies argmin γ → PPL, computes the scaleup multiplier vs the small-scale γ⋆ = 0.166, and writes:

- `GAMMA_SWEEP_RESULTS.md`  — γ→outcome table, γ⋆ identification, scaleup-multiplier interpretation, sensitivity diagnosis (PPL range)
- `gamma_sweep.png`         — left: val PPL vs γ with annotated points and dashed E5 γ⋆ reference line; right: train→val gap vs γ


In [ ]:
gamma_csv = ','.join(f'{g:.3f}' for g in GAMMAS)
subprocess.run([sys.executable, str(SCALEUP_DIR / 'aggregate_gamma_sweep_results.py'),
                '--results-dir', str(DRIVE_RESULTS),
                '--seed', str(SEED),
                '--gammas', gamma_csv,
                '--out-dir', str(DRIVE_RESULTS)],
               check=True)
print('\n--- GAMMA_SWEEP_RESULTS.md ---\n')
print((DRIVE_RESULTS / 'GAMMA_SWEEP_RESULTS.md').read_text())


In [ ]:
from IPython.display import Image, display
p = DRIVE_RESULTS / 'gamma_sweep.png'
if p.exists():
    print(p)
    display(Image(str(p)))
else:
    print('gamma_sweep.png missing - re-run aggregator')


## 5. Persistence & next steps

All artifacts live at:

```
/content/drive/MyDrive/semsimula_pilot/
├── results/
│   ├── splm_em_ln_scaleup_scaleup_g166_seed0_*
│   ├── splm_em_ln_scaleup_scaleup_g200_seed0_*
│   ├── splm_em_ln_scaleup_scaleup_g250_seed0_*
│   ├── splm_em_ln_scaleup_scaleup_g300_seed0_*
│   ├── splm_em_ln_scaleup_scaleup_g350_seed0_*
│   ├── GAMMA_SWEEP_RESULTS.md
│   └── gamma_sweep.png
└── logs/
    └── splm_em_ln_g{166,200,250,300,350}_<tag>_<timestamp>.log
```

### What to do with the results

1. Download `GAMMA_SWEEP_RESULTS.md` and `gamma_sweep.png` from Drive to your laptop.
2. Place them under `notebooks/conservative_arch/scaleup/results/gamma_sweep/` in the local repo.
3. Commit the markdown + plot.
4. Update `docs/Plan_for_first_TMLR_paper_v1.md` and the SPLM em_ln subsection of the Discussion to cite the γ⋆ at scaleup and the scaleup multiplier.
5. If multiplier ∈ [1.55, 2.05], the **1.8× scaleup-multiplier hypothesis** is confirmed and supports the broader claim that SPLM optimal γ scales with model depth/width.
6. If γ-sweep PPL range < 0.20 PPL, conclude that **γ is insensitive at scaleup** and that the colab_pilot Arm 2's `--fixed-gamma 0.30` is a reasonable but not uniquely-optimal choice.

### Adding intermediate γ values

Append to `GAMMAS` in the dispatch globals cell (e.g. `[0.166, 0.20, 0.225, 0.25, 0.275, 0.30, 0.325, 0.35]`) and re-run all cells.  Skip-if-done makes incremental refinement cheap.
